In [31]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report

In [32]:
df = pd.read_excel(r"C:\Users\admin\Downloads\Retail_Banking_Project\Merged_retail_banking.xlsx")

In [33]:
print(df.columns.tolist())

['customer_id', 'age', 'region', 'preferred_channel', 'monthly_transactions_avg', 'avg_monthly_balance', 'num_products_held', 'digital_adoption_score', 'credit_card_issued', 'loan_taken', 'Age_group', 'Balance_Category', 'month', 'monthly_transactions', 'Month_Name', 'Active customer']


In [35]:
# Convert text to lowercase first
df["Active customer"] = df["Active customer"].str.lower()

# Map values to 0 and 1
df["Active customer"] = df["Active customer"].map({
    "yes": 1,
    "no": 0
})

# Verify conversion
print(df["Active customer"].unique())

[1 0]


In [37]:
print(df["Active customer"].isnull().sum())

0


In [39]:
print(df["Active customer"].head())
print(df["Active customer"].dtype)
print(df["Active customer"].unique())

0    1
1    0
2    1
3    1
4    1
Name: Active customer, dtype: int64
int64
[1 0]


In [40]:
X = df.drop("Active customer", axis=1)
y = df["Active customer"]

In [41]:
columns_to_drop = []

if "customer_id" in X.columns:
    columns_to_drop.append("customer_id")

if "month" in X.columns:
    columns_to_drop.append("month")

X = X.drop(columns=columns_to_drop)

print("Dropped:", columns_to_drop)

Dropped: ['customer_id', 'month']


In [42]:
print(X.isnull().sum())
print(y.isnull().sum())

age                           0
region                        0
preferred_channel             0
monthly_transactions_avg      0
avg_monthly_balance           0
num_products_held             0
digital_adoption_score        0
credit_card_issued            0
loan_taken                    0
Age_group                   216
Balance_Category              0
monthly_transactions          0
Month_Name                    0
dtype: int64
0


In [43]:
# Fill missing values in Age_group
X["Age_group"] = X["Age_group"].fillna("Unknown")

In [44]:
print(X.isnull().sum())

age                         0
region                      0
preferred_channel           0
monthly_transactions_avg    0
avg_monthly_balance         0
num_products_held           0
digital_adoption_score      0
credit_card_issued          0
loan_taken                  0
Age_group                   0
Balance_Category            0
monthly_transactions        0
Month_Name                  0
dtype: int64


In [45]:
#Fill Missing Values
# Numerical columns
num_cols = X.select_dtypes(include=["int64","float64"]).columns

# Categorical columns
cat_cols = X.select_dtypes(include=["object"]).columns

# Fill missing values
X[num_cols] = X[num_cols].fillna(X[num_cols].median())

X[cat_cols] = X[cat_cols].fillna("Unknown")

In [46]:
#Identify Feature Types
numeric_features = X.select_dtypes(include=["int64","float64"]).columns

categorical_features = X.select_dtypes(include=["object"]).columns

print(numeric_features)
print(categorical_features)

Index(['age', 'monthly_transactions_avg', 'avg_monthly_balance',
       'num_products_held', 'digital_adoption_score', 'credit_card_issued',
       'loan_taken', 'monthly_transactions'],
      dtype='object')
Index(['region', 'preferred_channel', 'Age_group', 'Balance_Category',
       'Month_Name'],
      dtype='object')


In [47]:
#one hot encoding
preprocessor = ColumnTransformer(
    transformers=[
        (
            "cat",
            OneHotEncoder(handle_unknown="ignore"),
            categorical_features
        )
    ],
    remainder="passthrough"
)

In [48]:
#split data
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42
)

In [49]:
#Logistic Regression Model
model = Pipeline([
    ("preprocessor", preprocessor),
    ("classifier", LogisticRegression(max_iter=1000))
])

In [50]:
#Train Model
model.fit(X_train, y_train)

print("Model trained successfully.")

Model trained successfully.


D:\Users\admin\anaconda3\Lib\site-packages\sklearn\linear_model\_logistic.py:473: ConvergenceWarning: lbfgs failed to converge after 1000 iteration(s) (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT

Increase the number of iterations to improve the convergence (max_iter=1000).
You might also want to scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


In [51]:
predictions = model.predict(X_test)

print(predictions[:10])

[1 1 1 1 1 0 1 1 0 1]


In [52]:
accuracy = accuracy_score(y_test, predictions)

print("Accuracy:", accuracy)

Accuracy: 0.98625


In [53]:
print(classification_report(y_test, predictions))

              precision    recall  f1-score   support

           0       0.98      0.96      0.97      1062
           1       0.99      0.99      0.99      3738

    accuracy                           0.99      4800
   macro avg       0.98      0.98      0.98      4800
weighted avg       0.99      0.99      0.99      4800



Summary of this process:

-Loaded and explored the retail banking dataset.
-Converted the target variable (Active customer) from text (yes/No) to numeric (1/0).
-Removed unnecessary columns and handled missing values.
-Applied One-Hot Encoding to convert categorical features into numerical format.
-Split the dataset into 80% training and 20% testing sets.
-Built a Logistic Regression model using a preprocessing pipeline.
-Trained the model on the training data and generated predictions on the test data.
-Evaluated the model using Accuracy Score and Classification Report.
-Logistic Regression was chosen because it is suitable for predicting binary outcomes such as Active vs. Inactive Customer.

In [ ]:
Theory part :

Linear Regression vs Logistic Regression
Objective

To understand why Logistic Regression is the preferred model for predicting binary outcomes in banking, such as whether a customer is likely to default on a loan or remain an active customer.

1) Linear Regression
What is Linear Regression?

Linear Regression is a supervised machine learning algorithm used to predict continuous numerical values. It finds the best-fit line that represents the relationship between independent variables (features) and a dependent variable (target).

Output

The output can be any real number, such as:

4500.75
65000
-120

There is no restriction on the range of value

Banking Use Cases

Linear Regression is suitable for predicting numerical quantities such as:

Average monthly account balance
Monthly transaction amount
Customer spending
Loan amount
Annual income

For Example:

A bank wants to predict the average monthly balance of a customer based on age, income, and number of products held.

2) Limitations of Linear Regression for Classification

Linear Regression is not suitable for classification problems because:

It can predict values below 0 or above 1.
These outputs cannot be interpreted as probabilities.
It cannot directly classify customers into categories such as Default or Non-Default.

Example:

Suppose the model predicts:

1.45

or

-0.30

These values do not make sense as probabilities because probabilities must lie between 0 and 1.

3) Logistic Regression
What is Logistic Regression?

Logistic Regression is a supervised learning algorithm used for binary classification problems.

Instead of predicting a continuous value, it predicts the probability that an event will occur.

Output

The output always lies between:

0 and 1

For example:

Probability	Prediction
0.15	Non-Default
0.42	Non-Default
0.78	Default
0.95	Default

Typically, a threshold of 0.5 is used:

Probability ≥ 0.5 → Class 1
Probability < 0.5 → Class 0

4) Sigmoid Function

Logistic Regression uses the Sigmoid Function to convert any real-valued prediction into a probability between 0 and 1.

The sigmoid function produces an S-shaped curve.

Very large positive values approach 1
Very large negative values approach 0
A value of 0 maps to 0.5

This makes the output suitable for probability-based decision making.

5) Banking Use Cases for Logistic Regression

Logistic Regression is commonly used for predicting binary outcomes in banking, such as:

Loan Default vs Non-Default
High Risk vs Low Risk
Active Customer vs Inactive Customer
Fraudulent Transaction vs Genuine Transaction
Customer Churn vs Retained Customer
Credit Card Approval vs Rejection

Example:

If the model predicts a default probability of 0.82, the bank may classify the customer as high risk and perform additional credit checks before approving a loan.

6) Why Probability Estimation is Important

Banks need to understand how likely an event is to occur, not just assign a label.

For example:

Customer	Probability of Default
A	0.10
B	0.55
C	0.92

This allows the bank to make informed decisions, such as:

Approving or rejecting loans.
Adjusting interest rates based on risk.
Identifying customers who require closer monitoring.

Model Initialization & Training

In [21]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

In [5]:
df = pd.read_excel(r"C:\Users\admin\Downloads\Retail_Banking_Project\Merged_retail_banking.xlsx")

In [22]:
df.head()

,customer_id,age,region,preferred_channel,monthly_transactions_avg,avg_monthly_balance,num_products_held,digital_adoption_score,credit_card_issued,loan_taken,Age_group,Balance_Category,month,monthly_transactions,Month_Name,Active customer
0,CUST100007,56,South,ATM,23,37632,2,0.52,0,0,46-60,Medium,2022-01-01,15,January,yes
1,CUST100007,56,South,ATM,23,37632,2,0.52,0,0,46-60,Medium,2022-02-01,12,February,No
2,CUST100007,56,South,ATM,23,37632,2,0.52,0,0,46-60,Medium,2022-03-01,20,March,yes
3,CUST100007,56,South,ATM,23,37632,2,0.52,0,0,46-60,Medium,2022-04-01,21,April,yes
4,CUST100007,56,South,ATM,23,37632,2,0.52,0,0,46-60,Medium,2022-05-01,19,May,yes


In [23]:
print(df.isnull().sum())

customer_id                   0
age                           0
region                        0
preferred_channel             0
monthly_transactions_avg      0
avg_monthly_balance           0
num_products_held             0
digital_adoption_score        0
credit_card_issued            0
loan_taken                    0
Age_group                   216
Balance_Category              0
month                         0
monthly_transactions          0
Month_Name                    0
Active customer               0
dtype: int64


In [24]:
if "Age_group" in df.columns:
    df["Age_group"] = df["Age_group"].fillna("Unknown")

In [25]:
columns_to_drop = ["customer_id", "month", "Month_Name"]

for col in columns_to_drop:
    if col in df.columns:
        df.drop(col, axis=1, inplace=True)

In [28]:
X = df.drop("Active customer", axis=1)
y = df["Active customer"]

In [29]:
# Encoding Categorical Columns
categorical_columns = X.select_dtypes(include=["object"]).columns

X = pd.get_dummies(
    X,
    columns=categorical_columns,
    drop_first=True
)

In [30]:
#Scale Numerical Features
numerical_columns = X.select_dtypes(include=["int64", "float64"]).columns

scaler = StandardScaler()

X[numerical_columns] = scaler.fit_transform(X[numerical_columns])

In [ ]:
model = LogisticRegression(
    random_state=42,
    max_iter=1000
)

In [41]:
# Train model
from sklearn.linear_model import LogisticRegression

model = LogisticRegression(random_state=42, max_iter=1000)
model.fit(X_train, y_train)

,penalty,'l2'
,dual,False
,tol,0.0001
,C,1.0
,fit_intercept,True
,intercept_scaling,1
,class_weight,None
,random_state,42
,solver,'lbfgs'
,max_iter,1000
,multi_class,'deprecated'


In [42]:
# Create coefficients
coefficients = pd.DataFrame({
    "Feature": X_train.columns,
    "Coefficient": model.coef_[0]
})

coefficients = coefficients.sort_values(
    by="Coefficient",
    ascending=False
)

In [43]:
# Top Positive Features
print(coefficients.head(10))

                   Feature  Coefficient
7     monthly_transactions    22.732607
17           Age_group_60+     0.453294
16         Age_group_46-60     0.262861
9             region_North     0.191279
15         Age_group_31-45     0.181468
11             region_West     0.155667
8              region_East     0.139493
19    Balance_Category_Low     0.105706
10            region_South     0.083298
4   digital_adoption_score     0.058242


In [44]:
# Top Negative Features
print(coefficients.tail(10))

                             Feature  Coefficient
2                avg_monthly_balance     0.015650
6                         loan_taken     0.010627
20           Balance_Category_Medium    -0.037615
3                  num_products_held    -0.052462
1           monthly_transactions_avg    -0.053659
14  preferred_channel_Online Banking    -0.064318
13      preferred_channel_Mobile App    -0.093459
0                                age    -0.155263
12          preferred_channel_Branch    -0.164399
18                 Age_group_Unknown    -0.292587


In [45]:
coefficients.to_excel("Logistic_Regression_Coefficients.xlsx", index=False)